# Day 12 - Softmax Attention and Simplified FlashAttention-2

**Goal:** derive scaled dot-product attention, turn stable softmax into an
online recurrence, and implement the same one-pass forward algorithm in
Triton and CuTe DSL without materializing the full attention matrix.

This lesson follows the algorithm and implementation discussion in
[From FlashAttention to FlashAttention 2](https://supercharleszhu.github.io/posts/from-flashattention-to-flashattention-2/),
especially its explanations of Q-block ownership, delayed normalization, and
warp partitioning along Q rows.

## 1. What softmax attention computes

For one batch-head, let:

- `Q` have shape `(N_q, D)`;
- `K` and `V` have shape `(N_k, D)`;
- `D` be the head dimension.

Scaled dot-product attention is:

$$
S = \frac{QK^T}{\sqrt{D}},
\qquad
P_{ij} =
\frac{\exp(S_{ij})}
{\sum_{r=0}^{N_k-1}\exp(S_{ir})},
\qquad
O = PV.
$$

Each query row compares with every key row. Softmax converts those scores into
positive weights summing to one, and the output is a weighted combination of
the value rows.

The stable form subtracts each score row's maximum:

$$
m_i = \max_j S_{ij},
\qquad
P_{ij} =
\frac{\exp(S_{ij}-m_i)}
{\sum_r \exp(S_{ir}-m_i)}.
$$

Subtracting the maximum does not change the result because the same factor
$e^{-m_i}$ appears in every numerator and the denominator. It prevents large
positive scores from overflowing `exp`.

## 2. Why ordinary attention is memory-hungry

A direct implementation commonly creates:

```text
scores        = Q @ K.T       # N_q x N_k
probabilities = softmax(...)  # N_q x N_k
output        = probabilities @ V
```

The score/probability matrix is quadratic in sequence length. With
`N_q=N_k=N`, it contains `N^2` elements even though the final output contains
only `N*D`. Writing and rereading that matrix from HBM is the main problem
FlashAttention removes.

FlashAttention tiles K/V and keeps only a small score tile on chip:

```text
one Q row/block
    for each K/V tile:
        score tile = Q tile @ K tile.T
        update online softmax state
        update output accumulator with V tile
    normalize once and store O
```

The complete `N x N` matrix is never stored in HBM.

## 3. One-pass softmax attention equation

For one query row, maintain three values after processing some K/V tiles:

$$
m = \max_{\text{processed }j} s_j,
$$

$$
\ell = \sum_{\text{processed }j} e^{s_j-m},
$$

$$
U = \sum_{\text{processed }j} e^{s_j-m}v_j.
$$

`U` is deliberately **unnormalized**. The current attention output would be
`U / l`, but FlashAttention-2 delays that division until all tiles have been
processed.

For a new score tile $S_t$ and matching value tile $V_t$, compute:

$$
\widetilde{m}_t = \operatorname{rowmax}(S_t),
\qquad
m' = \max(m, \widetilde{m}_t),
$$

$$
\alpha = e^{m-m'},
\qquad
P_t = e^{S_t-m'},
$$

$$
\ell' = \alpha\ell + \operatorname{rowsum}(P_t),
$$

$$
U' = \alpha U + P_tV_t.
$$

After the final tile:

$$
O = \frac{U}{\ell}.
$$

These updates preserve the invariant:

$$
\ell' = \sum_{\text{all processed }j}e^{s_j-m'},
\qquad
U' = \sum_{\text{all processed }j}e^{s_j-m'}v_j.
$$

The factor `alpha` rescales all old terms from the old maximum to the new
maximum. This is why the algorithm remains numerically stable even when a
later tile contains a much larger score.

The normalized FA1 form updates `O` after each tile. The equations are
equivalent, but the unnormalized FA2 form avoids repeatedly dividing and
rescaling every output element:

$$
O' =
\frac{\alpha\ell}{\ell'}O
+ \frac{P_tV_t}{\ell'}.
$$

## 4. What FlashAttention-2 changes

The referenced deep dive highlights three changes from FlashAttention-1:

1. **Delay output normalization.** Accumulate `U` and divide once at the end,
   reducing non-matmul rescaling work.
2. **Assign each Q block to one CTA.** The CTA loops over all K/V blocks and
   owns its output, avoiding forward atomics.
3. **Partition warps only along Q rows.** Every warp sees the full K-column
   extent for its Q rows, so softmax does not require cross-warp exchange.

A production FA2 kernel also uses tensor-core MMA, swizzled shared memory,
asynchronous K/V pipelines, FP16/BF16 inputs, causal/edge masks, and a
specialized backward pass. This day isolates the forward algorithm before
those mechanisms.

## 5. Correctness and benchmark contract

The two teaching paths intentionally use different API shapes:

```text
Triton Q/K/V/O: [batch_heads, sequence_length, head_dim]
CuTe   Q/O:     [M, head_dim]
CuTe   K/V:     [N, head_dim]
```

The Triton path uses FP16 Q/K/V/output with FP32 online-softmax and output
accumulation. The CuTe path keeps Q/K/V/output storage in FP32, but
`MmaTF32Op` converts matrix-operand fragments to TF32 and executes both
`QK^T` and `PV` on tensor cores with FP32 accumulation.

Removing the batch-head dimension keeps the CuTe ownership lesson focused:
grid `x` enumerates 64-row Q tiles, and a four-warp CTA owns each tile. Each
warp owns 16 Q rows and traverses the complete K-column extent for those
rows. There is no grid `y` batch-head coordinate or batch-head term in global
indexing.

The reference explicitly materializes scores only for testing and always
computes it in FP32 before converting to the input dtype:

```python
scale = 1.0 / math.sqrt(head_dim)
scores = q.float() @ k.float().transpose(-1, -2) * scale
reference = (
    torch.softmax(scores, dim=-1) @ v.float()
).to(q.dtype)
```

Each implementation is immediately followed by its own correctness and
benchmark cells. Correctness includes sequence lengths 17, 33, and 64 so edge
tiles and partial Q blocks are exercised. Each benchmark:

- compiles and launches the implementation before timing;
- checks its output against the same PyTorch reference;
- uses `triton.testing.do_bench`;
- reports forward matmul FLOPs:

$$
\text{FLOPs} \approx 4 B_h N^2 D.
$$

The count includes `QK^T` and `PV` but omits exponentials, reductions, and
rescaling, so the reported TFLOP/s is an attention convention rather than a
complete instruction count.

## Equation checkpoint: test the invariant on ordinary tensors

Before writing a kernel, verify the update independently of GPU
ownership. The code below streams key/value tiles and compares
`(m, l, U)` with direct softmax. Notice that `U` is normalized only
after the loop.

In [ ]:
import math
import torch


def online_attention_reference(q, k, v, key_tile=16):
    scale = 1.0 / math.sqrt(q.shape[-1])
    running_max = torch.tensor(
        -torch.inf, device=q.device, dtype=torch.float32
    )
    running_sum = torch.tensor(
        0.0, device=q.device, dtype=torch.float32
    )
    output_accumulator = torch.zeros_like(q)

    for start in range(0, k.shape[0], key_tile):
        k_tile = k[start : start + key_tile]
        v_tile = v[start : start + key_tile]
        scores = (k_tile @ q) * scale
        new_max = torch.maximum(
            running_max, scores.max()
        )
        previous_scale = torch.exp(
            running_max - new_max
        )
        probabilities = torch.exp(scores - new_max)
        running_sum = (
            running_sum * previous_scale
            + probabilities.sum()
        )
        output_accumulator = (
            output_accumulator * previous_scale
            + probabilities @ v_tile
        )
        running_max = new_max

    return output_accumulator / running_sum


torch.manual_seed(0)
q_demo = torch.randn(32, device="cuda")
k_demo = torch.randn(37, 32, device="cuda")
v_demo = torch.randn(37, 32, device="cuda")
direct_demo = torch.softmax(
    k_demo @ q_demo / math.sqrt(32), dim=0
) @ v_demo
online_demo = online_attention_reference(
    q_demo, k_demo, v_demo
)
torch.testing.assert_close(
    online_demo, direct_demo, atol=2e-6, rtol=2e-5
)
print("online recurrence matches direct attention")

## Notebook setup

The implementations share only input validation, the direct
PyTorch oracle, deterministic input generation, and the FLOP
convention. Their correctness and timing functions remain
separate so each implementation is evaluated immediately after it
is introduced.

In [ ]:
import math

import torch

import cutlass
import cutlass.cute as cute
from cutlass._mlir.dialects import cute_nvgpu as _cute_nvgpu_ir
from cutlass.cute.atom import Trait, make_atom
from cutlass.cute.core import _pack_shape
from cutlass.cute.runtime import from_dlpack

cutlass.cuda.initialize_cuda_context()

WARP_SIZE = 32
WARPS_PER_CTA = 4
CTA_THREADS = WARP_SIZE * WARPS_PER_CTA
QUERY_ROWS_PER_WARP = 16
QUERY_TILE = QUERY_ROWS_PER_WARP * WARPS_PER_CTA
KEY_TILE = 64

def _validate_inputs(
    q,
    k,
    v,
    expected_dtype=None,
    expected_ndim: int = 3,
    allow_cross_attention: bool = False,
) -> None:
    import torch

    if not (q.is_cuda and k.is_cuda and v.is_cuda):
        raise ValueError("q, k, and v must be CUDA tensors")
    if k.dtype != q.dtype or v.dtype != q.dtype:
        raise TypeError("q, k, and v must use the same dtype")
    if expected_dtype is not None and q.dtype != expected_dtype:
        raise TypeError(f"q, k, and v must use {expected_dtype}")
    invalid_shape = (
        q.ndim != expected_ndim
        or k.ndim != expected_ndim
        or v.ndim != expected_ndim
    )
    if allow_cross_attention:
        invalid_shape = invalid_shape or (
            k.shape != v.shape
            or q.shape[:-2] != k.shape[:-2]
            or q.shape[-1] != k.shape[-1]
        )
    else:
        invalid_shape = (
            invalid_shape or k.shape != q.shape or v.shape != q.shape
        )
    if invalid_shape:
        shape_contract = (
            "[batch_heads, sequence_length, head_dim]"
            if expected_ndim == 3
            else "Q [M, D] and K/V [N, D]"
        )
        raise ValueError(
            f"invalid attention shapes; expected {shape_contract}"
        )
    if not (q.is_contiguous() and k.is_contiguous() and v.is_contiguous()):
        raise ValueError("q, k, and v must be contiguous")
    if any(
        dimension == 0
        for tensor in (q, k, v)
        for dimension in tensor.shape
    ):
        raise ValueError("attention dimensions must be non-zero")

def attention_reference(q, k, v):
    import torch

    scale = 1.0 / math.sqrt(q.shape[-1])
    scores = (
        torch.matmul(q.float(), k.float().transpose(-1, -2))
        * scale
    )
    output = torch.matmul(torch.softmax(scores, dim=-1), v.float())
    return output.to(q.dtype)

TEST_SHAPES = ((2, 17, 32), (3, 33, 64), (4, 64, 64))

BENCHMARK_SHAPES = ((16, 256, 64), (16, 512, 64))

CUTE_TEST_SHAPES = (
    (2, 3, 4),
    (17, 19, 32),
    (33, 35, 64),
    (64, 64, 64),
    (512, 256, 128),
)

CUTE_BENCHMARK_SHAPES = ((256, 64), (512, 64), (4096, 64))

def _make_inputs(shape, seed, dtype=None):
    import torch

    if dtype is None:
        dtype = torch.float32
    torch.manual_seed(seed)
    q = torch.randn(shape, device="cuda", dtype=dtype)
    k = torch.randn_like(q)
    v = torch.randn_like(q)
    return q, k, v

def attention_flops(
    batch_heads: int,
    sequence_length: int,
    head_dim: int,
) -> int:
    return (
        4
        * batch_heads
        * sequence_length
        * sequence_length
        * head_dim
    )

## 6. Implementations



Both implementations apply the same `(m, l, U)` invariant to matrix tiles.
They differ in storage dtype, tile shape, and how their tensor-core ownership
is expressed.

### 6.1 Triton implementation

This implementation follows the forward path in
[Triton Tutorial 06: Fused Attention](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html),
with a deliberately narrow feature set:

- FlashAttention-2 forward only;
- non-causal attention;
- FP16 Q/K/V and FP16 output;
- FP32 online-softmax and output accumulation;
- four warps, with no warp specialization;
- no FP8, tensor descriptors, autotuning, or backward kernel.

One program owns a block of 64 Q rows rather than one row. It keeps the Q tile
resident and processes K/V blocks of 32 or 64 rows:

```text
Q_tile: BLOCK_M x D
K_tile: BLOCK_N x D
V_tile: BLOCK_N x D

scores = Q_tile @ K_tile.T / sqrt(D)
m_new = max(m, rowmax(scores))
alpha = exp2(m - m_new)
p = exp2(scores - m_new[:, None])
l = alpha * l + rowsum(p)
U = alpha[:, None] * U + p.to(fp16) @ V_tile
m = m_new
```

`tl.dot` lowers both matrix products to tensor-core operations. `U` is a
`BLOCK_M x D` FP32 accumulator. The `BLOCK_M x BLOCK_N` score/probability tile
stays on chip, so the kernel still never creates an `N x N` output.

The kernel uses base-2 exponentiation like the official tutorial. Multiplying
the softmax scale by `log2(e)` preserves the same mathematical softmax while
mapping to `exp2`.

The notebook places `test_triton_attention()` and
`triton_benchmark_suite()` directly after the implementation.

In [ ]:
def build_triton_attention():
    """Define a forward-only FP16 FlashAttention-2 Triton kernel."""

    import triton
    import triton.language as tl

    @triton.jit
    def triton_flash_attention2_kernel(
        q_ptr,
        k_ptr,
        v_ptr,
        o_ptr,
        softmax_scale,
        N_CTX: tl.constexpr,
        HEAD_DIM: tl.constexpr,
        BLOCK_M: tl.constexpr,
        BLOCK_N: tl.constexpr,
    ):
        query_block = tl.program_id(0)
        batch_head = tl.program_id(1)
        query_indices = (
            query_block * BLOCK_M + tl.arange(0, BLOCK_M)
        )
        key_offsets = tl.arange(0, BLOCK_N)
        dimensions = tl.arange(0, HEAD_DIM)
        query_mask = query_indices < N_CTX
        tensor_base = batch_head * N_CTX * HEAD_DIM
        q_offsets = (
            tensor_base
            + query_indices[:, None] * HEAD_DIM
            + dimensions[None, :]
        )
        q = tl.load(
            q_ptr + q_offsets,
            mask=query_mask[:, None],
            other=0.0,
        )

        running_max = (
            tl.zeros((BLOCK_M,), dtype=tl.float32)
            - float("inf")
        )
        running_sum = tl.zeros(
            (BLOCK_M,), dtype=tl.float32
        ) + 1.0
        output_accumulator = tl.zeros(
            (BLOCK_M, HEAD_DIM), dtype=tl.float32
        )
        qk_scale = softmax_scale * 1.44269504

        for key_start in tl.range(0, N_CTX, BLOCK_N):
            key_indices = key_start + key_offsets
            key_mask = key_indices < N_CTX
            kv_offsets = (
                tensor_base
                + key_indices[:, None] * HEAD_DIM
                + dimensions[None, :]
            )
            keys = tl.load(
                k_ptr + kv_offsets,
                mask=key_mask[:, None],
                other=0.0,
            )
            scores = tl.dot(q, tl.trans(keys))
            scores = tl.where(
                key_mask[None, :],
                scores * qk_scale,
                -1.0e6,
            )

            tile_max = tl.max(scores, axis=1)
            new_max = tl.maximum(running_max, tile_max)
            probabilities = tl.math.exp2(
                scores - new_max[:, None]
            )
            previous_scale = tl.math.exp2(
                running_max - new_max
            )
            tile_sum = tl.sum(probabilities, axis=1)

            values = tl.load(
                v_ptr + kv_offsets,
                mask=key_mask[:, None],
                other=0.0,
            )
            output_accumulator *= previous_scale[:, None]
            output_accumulator = tl.dot(
                probabilities.to(tl.float16),
                values,
                output_accumulator,
            )
            running_sum = running_sum * previous_scale + tile_sum
            running_max = new_max

        output = output_accumulator / running_sum[:, None]
        tl.store(
            o_ptr + q_offsets,
            output.to(tl.float16),
            mask=query_mask[:, None],
        )

    return triton, triton_flash_attention2_kernel

_TRITON_MODULE = None
_TRITON_KERNEL = None

def launch_triton_attention(q, k, v, output) -> None:
    global _TRITON_MODULE, _TRITON_KERNEL

    import torch

    _validate_inputs(q, k, v, torch.float16)
    if output.shape != q.shape or output.dtype != q.dtype:
        raise ValueError("output must match q")
    if _TRITON_KERNEL is None:
        _TRITON_MODULE, _TRITON_KERNEL = build_triton_attention()
    batch_heads, sequence_length, head_dim = q.shape
    if head_dim not in (32, 64, 128):
        raise ValueError("Triton attention supports head_dim 32, 64, or 128")
    block_m = 64
    block_n = min(64, head_dim)
    grid = (
        _TRITON_MODULE.cdiv(sequence_length, block_m),
        batch_heads,
    )
    _TRITON_KERNEL[grid](
        q,
        k,
        v,
        output,
        1.0 / math.sqrt(head_dim),
        N_CTX=sequence_length,
        HEAD_DIM=head_dim,
        BLOCK_M=block_m,
        BLOCK_N=block_n,
        num_warps=4,
        num_stages=3,
    )

#### Triton correctness

Run the irregular and regular sequence-length cases immediately.
Lengths 17 and 33 exercise both partial K/V tiles and partially
populated 64-row Q blocks. The reference computes in FP32, then
converts to FP16 for the output contract.

In [ ]:
def test_triton_attention() -> None:
    import torch

    for index, shape in enumerate(TEST_SHAPES):
        q, k, v = _make_inputs(
            shape, 20261015 + index, torch.float16
        )
        reference = attention_reference(q, k, v)
        output = torch.empty_like(q)
        launch_triton_attention(q, k, v, output)
        torch.cuda.synchronize()
        torch.testing.assert_close(
            output, reference, atol=2e-2, rtol=2e-2
        )
    print("Day 12 FP16 Triton FlashAttention-2: correctness OK")

In [ ]:
test_triton_attention()

#### Triton benchmark

Each FP16 shape is checked against PyTorch again before timing.
The reported FLOPs count the two matrix multiplications and omit
softmax operations, matching the usual attention convention.

In [ ]:
def benchmark_triton_attention(
    batch_heads: int,
    sequence_length: int,
    head_dim: int,
    warmup: int = 25,
    repetitions: int = 100,
) -> float:
    import torch
    import triton

    shape = (batch_heads, sequence_length, head_dim)
    q, k, v = _make_inputs(shape, 20261015, torch.float16)
    output = torch.empty_like(q)
    launch_triton_attention(q, k, v, output)
    torch.cuda.synchronize()
    torch.testing.assert_close(
        output,
        attention_reference(q, k, v),
        atol=2e-2,
        rtol=2e-2,
    )
    return triton.testing.do_bench(
        lambda: launch_triton_attention(q, k, v, output),
        warmup=warmup,
        rep=repetitions,
    )

def triton_benchmark_suite() -> None:
    print("BH x N x D | Triton ms | Triton TFLOP/s")
    print("----------- | --------- | ---------------")
    for shape in BENCHMARK_SHAPES:
        elapsed_ms = benchmark_triton_attention(*shape)
        print(
            f"{shape[0]:2d}x{shape[1]:4d}x{shape[2]:2d} | "
            f"{elapsed_ms:9.4f} | "
            f"{attention_flops(*shape) / (elapsed_ms * 1e9):15.2f}"
        )

In [ ]:
triton_benchmark_suite()

### 6.2 CuTe implementation



The referenced FA2 head-dimension-64 SM80 kernel uses a `128 x 128` CTA tile,
four warps, and a warp layout stacked only along M:

```cpp
Flash_fwd_kernel_traits<64, 128, 128, 4, ...>
TiledMMA<Atom, Layout<Shape<_4, _1, _1>>, Tile<_64, _16, _16>>
```

The CuTe DSL lesson preserves that defining scheduling rule while using a
`64 x 64` tile:

```text
four-warp CTA
Q/O API shape: [M, D], K/V API shape: [N, D]
├── Q tile:      64 x D, loaded once
├── K/V tile:   64 x D, shared by all four warps
├── warp 0:     Q rows  0..15, all 64 K columns
├── warp 1:     Q rows 16..31, all 64 K columns
├── warp 2:     Q rows 32..47, all 64 K columns
├── warp 3:     Q rows 48..63, all 64 K columns
├── scores:     64 x 64 = TF32 TiledMma(Q, K^T)
└── U registers: 64 x D = persistent TF32 MMA accumulator
```

FA2 can use a larger tile because BF16 operands consume half as much shared
memory and its score/probability fragments remain in registers. This
intermediate TF32 lesson now keeps scores and online-softmax state in
registers, but still converts the final probability fragment through one FP32
shared-memory tile before `PV`. `64 x 64` keeps that representation practical
while adding the production kernel's four-warp row schedule.

The implementation is functional rather than stateful:

```python
@cute.kernel
def simplified_flash_attention2_kernel(...):
    ...


@cute.jit
def simplified_flash_attention2(m_q, m_k, m_v, m_o):
    query_length, head_dim = m_q.shape
    key_length, _ = m_k.shape
    ...
```

There is no attention class or `self`; compile-time dimensions come directly
from the 2D input tensor shapes. The launcher selects an `m16n8k4` TF32 atom
for `D=4 mod 8` and an `m16n8k8` atom otherwise.

Q/K/V remain FP32 in global and shared memory. Copying their FP32 partitions
into the fragments created by:

```python
mma_op = MmaTF32Op((16, 8, 8))
tiled_mma = cute.make_tiled_mma(
    mma_op, atom_layout_mnk=(4, 1, 1)
)
```

performs the TF32 operand conversion required by
`mma.sync.aligned.m16n8k8.row.col.f32.tf32.tf32.f32`. Both matrix products
therefore use tensor cores while their accumulators remain FP32.

CuTe DSL 4.4.2 does not export `cute.nvgpu.warp.MmaTF32Op`. The lesson
includes a small compatibility operation that constructs the same SM80 MMA
atom through `MmaAtomSM80Type`; newer releases use the public class:

```python
MmaTF32Op = getattr(
    cute.nvgpu.warp, "MmaTF32Op", MmaTF32CompatOp
)
```

For CTA thread `t`, the explicit hierarchy is:

```python
warp = t // 32
lane = t % 32
```

`atom_layout_mnk=(4, 1, 1)` places the four warp MMA atoms along M and none
along N or K. Warp `w` therefore owns Q rows `16*w : 16*(w+1)` and sees every
K column for those rows. This is the essential FA2 rule: softmax rows never
span warps, so no cross-warp reduction is required.

After `QK^T`, the FP32 score accumulator remains in registers. The raw
`m16n8` C fragment gives each lane two logical rows and two columns from each
8-column MMA atom. Across the eight N atoms in a 64-key tile, one lane owns
16 scores for each of its two rows:

```python
column = n_block * 8 + (lane % 4) * 2 + value
```

The four lanes sharing an MMA row reduce their local maxima and sums with
`quad_reduce_max` and `quad_reduce_sum`, matching FA2's `Allreduce<4>`. The
running maximum, denominator, and unnormalized output stay in registers. Each
K/V iteration updates:

$$
m' = \max(m, \operatorname{rowmax}(S_t)),
$$

$$
\ell' = e^{m-m'}\ell + \operatorname{rowsum}(e^{S_t-m'}),
$$

$$
U' = e^{m-m'}U
   + \operatorname{TF32MMA}(e^{S_t-m'}, V_t).
$$

The resulting probabilities are written once to FP32 shared memory and copied
into TF32 A fragments for `PV`. This is one remaining difference from FA2,
which converts and retiles `acc_s` directly into the `PV` A-register layout.
The persistent `U` tile stays in MMA C registers across the entire K/V loop:

```python
r_output.fill(0.0)
for each_kv_tile:
    r_output[row] *= previous_scale[row]
    r_output += tiled_mma(probabilities, values)
r_output[row] /= running_sum[row]
```

For `m16n8k8`, each lane owns four C values per output atom: two values from
row `lane // 4` and two from row `lane // 4 + 8`. The CTA row adds the
warp's offset:

```python
row = warp * 16 + lane // 4 + (accumulator // 2) * 8
```

That ownership determines which online-softmax scale applies to each output
register. The normalized register fragment is copied to shared memory only
once for the final coalesced output store. Partial Q and K/V tiles are
zero/mask padded, so sequence lengths need not be multiples of 64.

The notebook places `test_cute_attention()` and `cute_benchmark_suite()`
directly after the implementation.

In [ ]:
import math

import cutlass
import cutlass.cute as cute
from cutlass._mlir.dialects import cute_nvgpu as _cute_nvgpu_ir
from cutlass.cute.atom import Trait, make_atom
from cutlass.cute.core import _pack_shape
from cutlass.cute.runtime import from_dlpack

WARP_SIZE = 32
WARPS_PER_CTA = 4
CTA_THREADS = WARP_SIZE * WARPS_PER_CTA
QUERY_ROWS_PER_WARP = 16
QUERY_TILE = QUERY_ROWS_PER_WARP * WARPS_PER_CTA
KEY_TILE = 64

def _validate_inputs(
    q,
    k,
    v,
    expected_dtype=None,
    expected_ndim: int = 3,
    allow_cross_attention: bool = False,
) -> None:
    import torch

    if not (q.is_cuda and k.is_cuda and v.is_cuda):
        raise ValueError("q, k, and v must be CUDA tensors")
    if k.dtype != q.dtype or v.dtype != q.dtype:
        raise TypeError("q, k, and v must use the same dtype")
    if expected_dtype is not None and q.dtype != expected_dtype:
        raise TypeError(f"q, k, and v must use {expected_dtype}")
    invalid_shape = (
        q.ndim != expected_ndim
        or k.ndim != expected_ndim
        or v.ndim != expected_ndim
    )
    if allow_cross_attention:
        invalid_shape = invalid_shape or (
            k.shape != v.shape
            or q.shape[:-2] != k.shape[:-2]
            or q.shape[-1] != k.shape[-1]
        )
    else:
        invalid_shape = (
            invalid_shape or k.shape != q.shape or v.shape != q.shape
        )
    if invalid_shape:
        shape_contract = (
            "[batch_heads, sequence_length, head_dim]"
            if expected_ndim == 3
            else "Q [M, D] and K/V [N, D]"
        )
        raise ValueError(
            f"invalid attention shapes; expected {shape_contract}"
        )
    if not (q.is_contiguous() and k.is_contiguous() and v.is_contiguous()):
        raise ValueError("q, k, and v must be contiguous")
    if any(
        dimension == 0
        for tensor in (q, k, v)
        for dimension in tensor.shape
    ):
        raise ValueError("attention dimensions must be non-zero")

class MmaTF32CompatTrait(Trait):
    """Trait for the SM80 TF32 MMA atom missing from CuTe DSL 4.4.2."""

class MmaTF32CompatOp(cute.nvgpu.warp.WarpMmaOp):
    """Backport of warp.MmaTF32Op for CuTe DSL 4.4.2."""

    def __init__(self, shape_mnk: tuple[int, int, int]) -> None:
        if shape_mnk not in ((16, 8, 4), (16, 8, 8)):
            raise ValueError(
                "TF32 warp MMA shape must be (16, 8, 4) or (16, 8, 8)"
            )
        self.shape_mnk = shape_mnk

    def _make_trait(self, *, loc=None, ip=None, **kwargs):
        shape_mnk = _pack_shape(self.shape_mnk, loc=loc, ip=ip)
        atom_type = _cute_nvgpu_ir.MmaAtomSM80Type.get(
            shape_mnk.type.attribute,
            cutlass.TFloat32.mlir_type,
            cutlass.TFloat32.mlir_type,
            cutlass.Float32.mlir_type,
        )
        return MmaTF32CompatTrait(
            make_atom(atom_type, loc=loc, ip=ip)
        )

    def _verify_fragment_A(self, input, *, loc=None, ip=None):
        return True

    def _verify_fragment_B(self, input, *, loc=None, ip=None):
        return True

MmaTF32Op = getattr(
    cute.nvgpu.warp, "MmaTF32Op", MmaTF32CompatOp
)

def quad_reduce_sum(value: cutlass.Float32) -> cutlass.Float32:
    """Return a sum replicated within each four-lane MMA row group."""

    for offset in (2, 1):
        value += cute.arch.shuffle_sync_bfly(
            value, offset=offset, mask=-1, mask_and_clamp=31
        )
    return value

def quad_reduce_max(value: cutlass.Float32) -> cutlass.Float32:
    """Return a maximum replicated within each four-lane MMA row group."""

    for offset in (2, 1):
        value = cute.arch.fmax(
            value,
            cute.arch.shuffle_sync_bfly(
                value, offset=offset, mask=-1, mask_and_clamp=31
            ),
        )
    return value

@cute.kernel
def simplified_flash_attention2_kernel(
    g_q: cute.Tensor,
    g_k: cute.Tensor,
    g_v: cute.Tensor,
    g_o: cute.Tensor,
    tiled_mma: cute.TiledMma,
) -> None:
    query_length, head_dim = g_q.shape
    key_length, _ = g_k.shape
    mma_head_dim = cute.ceil_div(head_dim, 8) * 8
    softmax_scale = 1.0 / math.sqrt(head_dim)
    thread_idx, _, _ = cute.arch.thread_idx()
    query_block, _, _ = cute.arch.block_idx()
    lane = thread_idx % WARP_SIZE

    q_layout = cute.make_layout(
        (QUERY_TILE, head_dim),
        stride=(head_dim, 1),
    )
    k_layout = cute.make_layout(
        (KEY_TILE, head_dim),
        stride=(1, KEY_TILE),
    )
    probability_layout = cute.make_layout(
        (QUERY_TILE, KEY_TILE), stride=(KEY_TILE, 1)
    )
    value_layout = cute.make_layout(
        (mma_head_dim, KEY_TILE),
        stride=(1, mma_head_dim),
    )
    output_layout = cute.make_layout(
        (QUERY_TILE, mma_head_dim),
        stride=(mma_head_dim, 1),
    )
    smem = cutlass.utils.SmemAllocator()
    s_q = smem.allocate_tensor(cutlass.Float32, q_layout, 16)
    s_k = smem.allocate_tensor(cutlass.Float32, k_layout, 16)
    s_v = smem.allocate_tensor(
        cutlass.Float32, value_layout, 16
    )
    s_tile_output = smem.allocate_tensor(
        cutlass.Float32, output_layout, 16
    )
    s_probability_view = smem.allocate_tensor(
        cutlass.Float32, probability_layout, 16
    )

    tile_elements = QUERY_TILE * head_dim
    items_per_thread = cute.ceil_div(tile_elements, CTA_THREADS)
    value_tile_elements = KEY_TILE * mma_head_dim
    value_items_per_thread = cute.ceil_div(
        value_tile_elements, CTA_THREADS
    )
    for item in cutlass.range_constexpr(items_per_thread):
        linear_idx = thread_idx + item * CTA_THREADS
        if linear_idx < tile_elements:
            row = linear_idx // head_dim
            dimension = linear_idx % head_dim
            query_idx = query_block * QUERY_TILE + row
            if query_idx < query_length:
                s_q[row, dimension] = g_q[query_idx, dimension]
            else:
                s_q[row, dimension] = cutlass.Float32.zero

    cute.arch.sync_threads()

    thr_mma = tiled_mma.get_slice(thread_idx)

    t_q = thr_mma.partition_A(s_q)
    t_k = thr_mma.partition_B(s_k)
    t_scores = thr_mma.partition_C(s_probability_view)
    r_q = tiled_mma.make_fragment_A(t_q)
    r_k = tiled_mma.make_fragment_B(t_k)
    r_scores = tiled_mma.make_fragment_C(t_scores)

    t_probabilities = thr_mma.partition_A(s_probability_view)
    t_v = thr_mma.partition_B(s_v)
    t_tile_output = thr_mma.partition_C(s_tile_output)
    r_probabilities = tiled_mma.make_fragment_A(t_probabilities)
    r_v = tiled_mma.make_fragment_B(t_v)
    r_output = tiled_mma.make_fragment_C(t_tile_output)
    r_output.fill(0.0)
    r_running_max = cute.make_rmem_tensor(
        (2,), cutlass.Float32
    )
    r_running_sum = cute.make_rmem_tensor(
        (2,), cutlass.Float32
    )
    for row_slot in cutlass.range_constexpr(2):
        r_running_max[row_slot] = -cutlass.Float32.inf
        r_running_sum[row_slot] = cutlass.Float32.zero

    score_k_blocks = cute.size(r_q, mode=[2])
    output_k_blocks = cute.size(r_probabilities, mode=[2])
    output_n_blocks = cute.size(r_output, mode=[2])
    for k_block in cutlass.range_constexpr(score_k_blocks):
        cute.autovec_copy(
            t_q[None, None, k_block],
            r_q[None, None, k_block],
        )

    key_tiles = cute.ceil_div(key_length, KEY_TILE)
    for key_tile in cutlass.range(key_tiles):
        key_start = key_tile * KEY_TILE
        for item in cutlass.range_constexpr(items_per_thread):
            linear_idx = thread_idx + item * CTA_THREADS
            if linear_idx < tile_elements:
                key_offset = linear_idx // head_dim
                dimension = linear_idx % head_dim
                key_idx = key_start + key_offset
                if key_idx < key_length:
                    s_k[key_offset, dimension] = g_k[
                        key_idx, dimension
                    ]
                else:
                    s_k[
                        key_offset, dimension
                    ] = cutlass.Float32.zero
        for item in cutlass.range_constexpr(
            value_items_per_thread
        ):
            linear_idx = thread_idx + item * CTA_THREADS
            if linear_idx < value_tile_elements:
                key_offset = linear_idx // mma_head_dim
                dimension = linear_idx % mma_head_dim
                key_idx = key_start + key_offset
                if key_idx < key_length and dimension < head_dim:
                    s_v[dimension, key_offset] = g_v[
                        key_idx, dimension
                    ]
                else:
                    s_v[
                        dimension, key_offset
                    ] = cutlass.Float32.zero
        cute.arch.sync_threads()

        r_scores.fill(0.0)
        for k_block in cutlass.range_constexpr(score_k_blocks):
            cute.autovec_copy(
                t_k[None, None, k_block],
                r_k[None, None, k_block],
            )
            cute.gemm(
                tiled_mma,
                r_scores,
                r_q[None, None, k_block],
                r_k[None, None, k_block],
                r_scores,
            )
        lane_in_quad = lane % 4
        for row_slot in cutlass.range_constexpr(2):
            accumulator_base = row_slot * 2
            thread_max = -cutlass.Float32.inf
            for n_block in cutlass.range_constexpr(
                cute.size(r_scores, mode=[2])
            ):
                for value in cutlass.range_constexpr(2):
                    column = (
                        n_block * 8 + lane_in_quad * 2 + value
                    )
                    key_idx = key_start + column
                    score = (
                        r_scores[
                            accumulator_base + value, 0, n_block
                        ]
                        * softmax_scale
                    )
                    if key_idx >= key_length:
                        score = -cutlass.Float32.inf
                    r_scores[
                        accumulator_base + value, 0, n_block
                    ] = score
                    thread_max = cute.arch.fmax(
                        thread_max, score
                    )
            tile_max = quad_reduce_max(thread_max)
            old_max = r_running_max[row_slot]
            new_max = cute.arch.fmax(old_max, tile_max)
            previous_scale = cute.math.exp(old_max - new_max)
            thread_sum = cutlass.Float32.zero
            for n_block in cutlass.range_constexpr(
                cute.size(r_scores, mode=[2])
            ):
                for value in cutlass.range_constexpr(2):
                    probability = cute.math.exp(
                        r_scores[
                            accumulator_base + value, 0, n_block
                        ]
                        - new_max
                    )
                    r_scores[
                        accumulator_base + value, 0, n_block
                    ] = probability
                    thread_sum += probability
            tile_sum = quad_reduce_sum(thread_sum)
            r_running_max[row_slot] = new_max
            r_running_sum[row_slot] = (
                r_running_sum[row_slot] * previous_scale
                + tile_sum
            )
            for n_block in cutlass.range_constexpr(
                output_n_blocks
            ):
                for value in cutlass.range_constexpr(2):
                    r_output[
                        accumulator_base + value, 0, n_block
                    ] = (
                        r_output[
                            accumulator_base + value, 0, n_block
                        ]
                        * previous_scale
                    )
        cute.copy(
            cute.make_copy_atom(
                cute.nvgpu.CopyUniversalOp(), cutlass.Float32
            ),
            r_scores,
            t_scores,
        )
        cute.arch.sync_threads()
        for k_block in cutlass.range_constexpr(output_k_blocks):
            cute.autovec_copy(
                t_probabilities[None, None, k_block],
                r_probabilities[None, None, k_block],
            )
            cute.autovec_copy(
                t_v[None, None, k_block],
                r_v[None, None, k_block],
            )
            cute.gemm(
                tiled_mma,
                r_output,
                r_probabilities[None, None, k_block],
                r_v[None, None, k_block],
                r_output,
            )
        cute.arch.sync_threads()

    for n_block in cutlass.range_constexpr(output_n_blocks):
        for accumulator in cutlass.range_constexpr(4):
            r_output[accumulator, 0, n_block] = (
                r_output[accumulator, 0, n_block]
                / r_running_sum[accumulator // 2]
            )
    cute.copy(
        cute.make_copy_atom(
            cute.nvgpu.CopyUniversalOp(), cutlass.Float32
        ),
        r_output,
        t_tile_output,
    )
    cute.arch.sync_threads()

    for item in cutlass.range_constexpr(items_per_thread):
        linear_idx = thread_idx + item * CTA_THREADS
        if linear_idx < tile_elements:
            row = linear_idx // head_dim
            dimension = linear_idx % head_dim
            query_idx = query_block * QUERY_TILE + row
            if query_idx < query_length:
                g_o[query_idx, dimension] = s_tile_output[row, dimension]

@cute.jit
def simplified_flash_attention2(
    m_q: cute.Tensor,
    m_k: cute.Tensor,
    m_v: cute.Tensor,
    m_o: cute.Tensor,
) -> None:
    if cutlass.const_expr(
        m_q.element_type != cutlass.Float32
        or m_k.element_type != cutlass.Float32
        or m_v.element_type != cutlass.Float32
        or m_o.element_type != cutlass.Float32
    ):
        raise TypeError(
            "Day 12 CuTe uses Float32 Q/K/V/output storage"
        )

    query_length, head_dim = m_q.shape
    key_length, key_head_dim = m_k.shape
    value_length, value_head_dim = m_v.shape
    output_length, output_head_dim = m_o.shape
    if cutlass.const_expr(
        key_length != value_length
        or head_dim != key_head_dim
        or head_dim != value_head_dim
        or query_length != output_length
        or head_dim != output_head_dim
    ):
        raise ValueError(
            "expected Q/O [M, D] and K/V [N, D]"
        )
    if cutlass.const_expr(head_dim > 128 or head_dim % 4 != 0):
        raise ValueError(
            "the TF32 TiledMMA lesson requires head_dim <= 128 "
            "and divisible by 4"
        )

    if cutlass.const_expr(head_dim % 8 == 0):
        mma_op = MmaTF32Op((16, 8, 8))
    else:
        mma_op = MmaTF32Op((16, 8, 4))
    tiled_mma = cute.make_tiled_mma(
        mma_op, atom_layout_mnk=(WARPS_PER_CTA, 1, 1)
    )
    simplified_flash_attention2_kernel(
        m_q, m_k, m_v, m_o, tiled_mma
    ).launch(
        grid=(cute.ceil_div(query_length, QUERY_TILE), 1, 1),
        block=(CTA_THREADS, 1, 1),
    )

def prepare_cute_attention(q, k, v):
    """Compile the 2D CuTe kernel and return stable runtime arguments."""

    import torch

    _validate_inputs(
        q,
        k,
        v,
        torch.float32,
        expected_ndim=2,
        allow_cross_attention=True,
    )
    output = torch.empty_like(q)
    q_cute = from_dlpack(q, assumed_align=16)
    k_cute = from_dlpack(k, assumed_align=16)
    v_cute = from_dlpack(v, assumed_align=16)
    output_cute = from_dlpack(output, assumed_align=16)
    compiled = cute.compile(
        simplified_flash_attention2,
        q_cute,
        k_cute,
        v_cute,
        output_cute,
    )
    return output, compiled, (q_cute, k_cute, v_cute, output_cute)

#### CuTe correctness

The CuTe API intentionally removes the batch-head axis: Q/O have
shape `[M, head_dim]`, while K/V have shape `[N, head_dim]`.
Different M and N values exercise partial 64-row Q tiles and
partial 64-key K/V tiles. The `M=512, N=256` case verifies that
the launch grid covers every query row when M is larger than N;
D=4 exercises the TF32 `m16n8k4` atom.
Four warps share each K/V tile and independently own 16 Q rows.
Storage remains FP32, while `MmaTF32Op` converts operand
fragments to TF32 for both tensor-core matrix products.

In [ ]:
def test_cute_attention() -> None:
    import torch

    for index, (query_length, key_length, head_dim) in enumerate(
        CUTE_TEST_SHAPES
    ):
        torch.manual_seed(20261015 + index)
        q = torch.randn(
            (query_length, head_dim),
            device="cuda",
            dtype=torch.float32,
        )
        k = torch.randn(
            (key_length, head_dim),
            device="cuda",
            dtype=torch.float32,
        )
        v = torch.randn_like(k)
        reference = attention_reference(q, k, v)
        output, compiled, arguments = prepare_cute_attention(q, k, v)
        compiled(*arguments)
        torch.cuda.synchronize()
        torch.testing.assert_close(
            output, reference, atol=3e-2, rtol=3e-2
        )
    print("Day 12 FP32/TF32 CuTe attention: correctness OK")

In [ ]:
test_cute_attention()

#### CuTe benchmark

Correctness is rechecked before timing each single-head `[N, D]`
FP32-storage, TF32-compute shape.

In [ ]:
def benchmark_cute_attention(
    sequence_length: int,
    head_dim: int,
    warmup: int = 25,
    repetitions: int = 100,
) -> float:
    import torch
    import triton

    shape = (sequence_length, head_dim)
    q, k, v = _make_inputs(shape, 20261015)
    output, compiled, arguments = prepare_cute_attention(q, k, v)
    compiled(*arguments)
    torch.cuda.synchronize()
    torch.testing.assert_close(
        output,
        attention_reference(q, k, v),
        atol=3e-2,
        rtol=3e-2,
    )
    return triton.testing.do_bench(
        lambda: compiled(*arguments),
        warmup=warmup,
        rep=repetitions,
    )

def benchmark_naive_attention(
    sequence_length: int,
    head_dim: int,
    warmup: int = 25,
    repetitions: int = 100,
) -> float:
    import torch
    import triton

    q, k, v = _make_inputs(
        (sequence_length, head_dim), 20261015
    )
    previous_precision = torch.get_float32_matmul_precision()
    torch.set_float32_matmul_precision("high")
    try:
        output = attention_reference(q, k, v)
        torch.cuda.synchronize()
        assert output.shape == q.shape
        return triton.testing.do_bench(
            lambda: attention_reference(q, k, v),
            warmup=warmup,
            rep=repetitions,
        )
    finally:
        torch.set_float32_matmul_precision(previous_precision)

def cute_benchmark_suite() -> None:
    print(
        "N x D | Naive ms | Naive TFLOP/s | "
        "CuTe ms | CuTe TFLOP/s | CuTe speedup"
    )
    print(
        "----- | -------- | ------------- | "
        "------- | ------------- | ------------"
    )
    for shape in CUTE_BENCHMARK_SHAPES:
        naive_ms = benchmark_naive_attention(*shape)
        cute_ms = benchmark_cute_attention(*shape)
        flops = attention_flops(1, *shape)
        print(
            f"{shape[0]:4d}x{shape[1]:2d} | "
            f"{naive_ms:8.4f} | "
            f"{flops / (naive_ms * 1e9):13.2f} | "
            f"{cute_ms:7.4f} | "
            f"{flops / (cute_ms * 1e9):13.2f} | "
            f"{naive_ms / cute_ms:12.2f}x"
        )

In [ ]:
cute_benchmark_suite()

## 7. Compare the implementations



Triton and CuTe both process 64 Q rows and up to 64 keys per update. Triton
uses FP16 tensor-core `tl.dot`; CuTe uses FP32 storage and TF32 `TiledMma`.
CuTe keeps scores, row reductions, and persistent output in registers. Only
the probability fragment still passes through shared memory before `PV`.
Neither implementation writes the full score or probability matrix to HBM.

Representative Triton B300 measurements retain 16 batch-heads:

| `BH x N x D` | Triton ms | Triton TFLOP/s |
|---|---:|---:|
| `16 x 256 x 64` | 0.0102 | 26.23 |
| `16 x 512 x 64` | 0.0123 | 87.34 |

The single-head comparison also measures the direct PyTorch expression:

```python
torch.softmax((q @ k.T) / math.sqrt(head_dim), dim=-1) @ v
```

FP32 matmul precision is set to `high`, allowing the same TF32 tensor-core
arithmetic category as the CuTe kernel. This implementation is mathematically
naive because it materializes the complete score and probability matrices,
but its two matrix multiplications use highly optimized cuBLAS kernels.

| `N x D` | Naive ms | Naive TFLOP/s | CuTe ms | CuTe TFLOP/s | CuTe speedup |
|---|---:|---:|---:|---:|---:|
| `256 x 64` | 0.0210 | 0.80 | 0.0408 | 0.41 | 0.52x |
| `512 x 64` | 0.0215 | 3.12 | 0.0719 | 0.93 | 0.30x |
| `4096 x 64` | 0.1054 | 40.75 | 0.4997 | 8.59 | 0.21x |

The four-warp `64 x 64` schedule plus register softmax is about 23-24% faster
than the preceding one-warp `16 x 16` version at the two smaller shapes.
Against the immediately preceding four-warp shared-score version, register
softmax improves latency by about 8% at `256 x 64`, 10% at `512 x 64`, and
11% at `4096 x 64`.

The teaching kernel is still about 1.9x slower at `256 x 64`, 3.3x slower at
`512 x 64`, and 4.7x slower at `4096 x 64`. A single head launches only four,
eight, or 64 CTAs respectively, so even the largest case does not fill all
148 B300 SMs in one wave. Production attention exposes batch and head
dimensions to the launch grid, and FA2 additionally retiles probabilities
directly in registers, pipelines asynchronous copies, and uses swizzled
lower-precision shared storage.

Run:

```bash
python day12_softmax_attention/solution.py
python day12_softmax_attention/solution.py --benchmark
```

## 8. Nsight Compute: why fused CuTe is slower

The benchmark establishes the application result:

```text
N=512, D=64
naive PyTorch: 0.0215 ms
four-warp CuTe: 0.0719 ms
```

The CuTe kernel is about `3.3x` slower even though it avoids materializing the
complete score and probability matrices. Nsight Compute explains why.

### 8.1 Reproduce the profile

`profile_target.py` compiles and warms up first, checks correctness, then uses
`cudaProfilerStart/Stop` around exactly one measured invocation. The naïve
region contains its four GPU kernels; the CuTe region contains one fused
kernel.

On the B300 pod:

```bash
cd /tmp/gpu-programming-puzzle-cute-dsl/day12_softmax_attention
PYTHON_BIN=../.venv/bin/python \
SEQUENCE_LENGTH=512 HEAD_DIM=64 \
bash collect_profiles.sh
```

The script creates raw CSV and `.ncu-rep` files under `reports/`. Open the
reports in the Nsight Compute UI for source and SASS inspection. The compact
checked-in snapshot is `profile_summary.csv`.

NCU replays kernels to collect counters, so its durations are perturbed.
Use the normal benchmark for final latency and NCU for relative evidence.
The profiled durations preserve the same result:

```text
naive stages total: 31.040 us
fused CuTe kernel: 110.496 us
ratio: 3.56x
```

### 8.2 Counter snapshot

The naïve expression launches four kernels:

| Path/stage | Grid blocks | Duration (NCU) | SM throughput | Tensor active | Registers/thread | Shared/block |
|---|---:|---:|---:|---:|---:|---:|
| CuTe fused | 8 | 110.496 us | 0.67% | 0.37% | 255 | 82.9 KB |
| Naïve `QK^T` | 64 | 7.392 us | 5.23% | 2.83% | 88 | 50.2 KB |
| Naïve scale | 256 | 5.024 us | 1.80% | 0% | 32 | 1.0 KB |
| Naïve softmax | 128 | 6.880 us | 6.53% | 0% | 48 | 1.0 KB |
| Naïve `PV` | 16 | 11.744 us | 1.63% | 0.89% | 69 | 227.3 KB |

The B300 has 148 SMs. For `M=512`, the fused kernel launches only:

```text
ceil(512 / 64) = 8 CTAs
```

At most eight SMs receive work in that wave, only 5.4% of the GPU. Each CTA
then serially processes eight 64-key tiles because online softmax makes the
K/V loop sequential. Production attention obtains enough parallel CTAs from
batch and head dimensions; this teaching API deliberately removed those
dimensions.

By comparison, the naïve stages expose grids of 64, 256, 128, and 16 blocks.
Although they launch separately, cuBLAS/CUTLASS and PyTorch spread each stage
over much more of the GPU.

### 8.3 Occupancy is constrained by the teaching representation

The CuTe kernel uses:

```text
255 registers/thread
82.944 KB shared memory/CTA
4 warps/CTA
```

On this B300, registers limit an SM to two resident CTAs. That is at most
eight resident warps per participating SM before other limits. NCU reports
only 6.24% active warps and 0.49% issue-active cycles.

Register softmax pushes the kernel to the 255-register ceiling and introduces
3,712 local-load sectors and 640 local-store sectors. Keeping row state in
shared memory removes those spills but is slower, so the checked-in version
accepts this small spill cost. The dominant limitation remains reduced
residency and almost no grid-level parallelism.

### 8.4 The remaining shared-memory path is still expensive

Register softmax removes the score store and reload. The kernel still stages
Q/K/V with simple layouts and writes probabilities to shared memory before
reloading them for `PV`. These unswizzled paths produce:

```text
shared-load bank conflicts:  261,341
shared-store bank conflicts: 281,600
total:                       542,941
```

The four naïve library kernels report only 140 shared-memory conflicts in
total. The CuTe stall distribution matches the shared-memory-heavy source:

| Stall reason | CuTe |
|---|---:|
| Short scoreboard, including shared/MIO dependencies | 32.86% |
| Long scoreboard, including L1TEX/global dependencies | 39.06% |
| Fixed-latency wait | 14.60% |
| CTA barrier | 1.33% |

The kernel is therefore not mainly waiting at `sync_threads()`. It is waiting
for data dependencies created by synchronous global/shared copies and the
remaining probability shared-memory path. NCU also reports zero `LDGSTS`
instructions, confirming that this version has no `cp.async` pipeline. By
contrast, the naïve `QK^T` GEMM executes 5,120 `LDGSTS` instructions.

### 8.5 Tensor cores are present but poorly utilized

The fused kernel executes tensor-core MMA for both matrix products, but the
GPU-wide tensor pipeline is active for only 0.37% of elapsed cycles. The
optimized naïve `QK^T` GEMM reaches 2.83% despite the small problem.

NCU also reports:

```text
CuTe tensor-pipe instructions:       32,768
naïve GEMM tensor-pipe instructions: 16,896
```

The teaching kernel builds both products from fine-grained `m16n8` warp MMA
atoms. The library GEMMs use tuned larger tiles; the `PV` kernel is an SM100
two-SM TMA kernel. The libraries perform the same mathematical products with
coarser instruction granularity, optimized epilogues, and architecture-aware
copy pipelines.

### 8.6 Fusion saves traffic, but traffic is not the bottleneck

Summing the NCU rows:

| Path | DRAM reads | L2 bytes |
|---|---:|---:|
| CuTe fused | 0.44 MB | 2.23 MB |
| Naïve four stages | 3.64 MB | 14.23 MB |

Fusion reduces L2 traffic by about `6.4x` and DRAM reads by about `8.3x`.
Nevertheless, the basic profile reports less than 3% DRAM throughput for
every naïve stage and 0.05% for CuTe. The `512 x 512` intermediates are small
enough to benefit heavily from B300 cache, so avoiding them does not offset
low launch parallelism and inefficient on-chip movement.

The evidence chain is:

```text
single-head API
  -> only 8 fused CTAs for M=512
  -> most of 148 SMs idle
  -> 255 registers + 83 KB shared memory constrain residency
  -> unswizzled Q/K/V/P paths create 543K shared conflicts
  -> scoreboard stalls dominate and tensor utilization stays below 1%
  -> tuned multi-kernel PyTorch is faster despite moving more bytes
```

## 9. Next optimization steps

Advance one mechanism at a time:

1. Implement FA2's `convert_layout_acc_Aregs` equivalent for TF32 CuTe DSL,
   converting and retiling register probabilities directly into the `PV` A
   operand without the remaining shared-memory tile.
2. Add FA2-style swizzled Q/K/V layouts to reduce the 543K shared bank
   conflicts.
3. Add 128-bit `cp.async` global-to-shared copies, then double-buffer K/V to
   overlap the next tile's loads with current-tile MMA.
4. Expand the four-warp tile from `64 x 64` toward FA2's `128 x 128` shape
   after removing the FP32 probability shared tile.
5. Restore batch/head launch dimensions so production-sized workloads expose
   enough CTAs to fill the GPU.
6. Add causal masks, logsumexp storage, and backward recomputation.
7. Treat TMA, warp specialization, and architecture-specific tensor-memory
   paths as separate advanced steps.